# Bài 02 · Hàm kích hoạt: từ công tắc bậc thang đến SwiGLU, trên mạng MNIST thật

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/02-ham-kich-hoat.ipynb)

Notebook đi kèm [Bài 02 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/) (reel và demo tương tác nằm ở trang bài học).
Sau tổng có trọng số $z = xW + b$, mỗi nơ-ron đưa $z$ qua một **hàm kích hoạt** $f$. Ta tự viết bằng numpy các hàm
đó cùng đạo hàm của chúng, rồi đặt từng hàm vào mạng MNIST 784 → 128 → 64 → 10 của khóa học để xem $f$ quyết định
những gì: mạng có thật sự “sâu” không, có học được không, và học nhanh đến đâu.

**Bạn sẽ làm:**
1. Viết sáu hàm kích hoạt và đạo hàm $f'$ của chúng, kiểm tra bằng sai phân hữu hạn.
2. Thấy bằng số rằng không có $f$, ba lớp gộp lại thành một ma trận, và vì sao hàm bậc thang không cho gradient.
3. Đo vùng bão hòa của sigmoid và tanh, rồi đưa $z$ thật của mạng MNIST (huấn luyện lại, 97,27%) qua từng hàm.
4. Huấn luyện lại cùng mạng với từng hàm, ba seed mỗi hàm, và đọc kết quả như một thí nghiệm.
5. Dựng một lớp SwiGLU, “MLP có cổng” của Llama 3, Qwen3 và DeepSeek-V3, và đếm tham số của nó.

**Cần biết trước:** [Bài 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/perceptron/)
(một nơ-ron, $z = xW + b$, hàm bậc thang). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Ký hiệu theo chương *Neural Computation* của [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).
Mọi con số trong notebook đều do chính các ô bên dưới tính ra. Những số mà demo và reel cũng hiển thị
($\sigma'(0) = 0{,}25$, mạng tham chiếu 97,27%…) đều trùng; các ô `assert` kiểm tra điều đó.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi. `sci()` in số rất nhỏ
dạng $9{,}1 \cdot 10^{-13}$.

In [ ]:
import gzip
import hashlib
import itertools
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (nét liền/gạch, hình dạng, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 9,1·10⁻¹³."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{vn(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(vn(13007, 0), vn(0.1, 1), pct(0.9727), sci(0.25**20))

## 2 · Dữ liệu MNIST

Cùng dữ liệu với Bài 01: 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28
(LeCun và cộng sự, 1998). Ô dưới tải 4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.
Mỗi ảnh được trải thành một vectơ hàng 784 số trong $[0, 1]$.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # mỗi hàng là một ảnh: x cỡ 1 × 784
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
print(f"huấn luyện: {vn(len(Xtr), 0)} ảnh {Xtr.shape}  ·  kiểm tra: {vn(len(Xte), 0)} ảnh {Xte.shape}")

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, img, lab in zip(axes, test_img, test_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
plt.show()

Đó là 10 ảnh kiểm tra đầu tiên với nhãn đúng ở trên. Ảnh đầu tiên, số 7, sẽ quay lại ở mục 8.

## 3 · Sáu hàm kích hoạt và đạo hàm của chúng

Một nơ-ron tính $z = xW + b$ rồi cho ra $f(z)$. Khi học bằng gradient (Bài 05), tín hiệu sửa lỗi đi ngược qua nơ-ron
và bị nhân với **đạo hàm** $f'(z)$. Nên mỗi hàm được viết kèm đạo hàm của nó:

$$\text{bậc thang: } f(z) = [z \ge 0] \qquad \sigma(z) = \frac{1}{1+e^{-z}} \qquad \tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

$$\operatorname{ReLU}(z) = \max(0, z) \qquad \operatorname{GELU}(z) = z\,\Phi(z) \qquad \operatorname{SiLU}(z) = z\,\sigma(z)$$

$\Phi$ là hàm phân phối tích lũy của phân phối chuẩn. Như GPT-2 và demo, ta tính GELU bằng dạng xấp xỉ
$\tfrac12 z\,(1 + \tanh[\sqrt{2/\pi}\,(z + 0{,}044715\,z^3)])$ của Hendrycks & Gimpel (mục 7 so với dạng chính xác).
Sigmoid được viết là $\tfrac12(1 + \tanh(z/2))$: cùng một hàm, nhưng không tràn số khi $z$ rất âm.

In [ ]:
C_GELU = np.sqrt(2 / np.pi)
NAMES = {
    "linear": "không có f",
    "step": "bậc thang",
    "sigmoid": "sigmoid",
    "tanh": "tanh",
    "relu": "ReLU",
    "gelu": "GELU",
    "silu": "SiLU",
    "logreg": "softmax 784 → 10",
}


def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))  # = 1 / (1 + e^−z)


def act(name, z):
    """Trả về f(z) và đạo hàm f′(z), cùng cỡ với z. `name` cũng có thể là một hàm tự viết (mục 9)."""
    if callable(name):
        return name(z)
    z = np.asarray(z, float)
    if name == "linear":  # không có hàm kích hoạt: f(z) = z
        return z, np.ones_like(z)
    if name == "step":  # hàm của perceptron: đạo hàm bằng 0 ở mọi nơi (không xác định tại 0)
        return (z >= 0).astype(float), np.zeros_like(z)
    if name == "sigmoid":
        s = sigmoid(z)
        return s, s * (1 - s)
    if name == "tanh":
        t = np.tanh(z)
        return t, 1 - t**2
    if name == "relu":  # quy ước f′(0) = 0, như PyTorch và d2l.ai
        return np.maximum(0, z), (z > 0).astype(float)
    if name == "gelu":  # dạng xấp xỉ tanh, như GPT-2 và demo
        z2 = z * z
        t = np.tanh(C_GELU * z * (1 + 0.044715 * z2))
        return 0.5 * z * (1 + t), 0.5 * (1 + t) + 0.5 * z * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * z2)
    if name == "silu":
        s = sigmoid(z)
        return z * s, s * (1 + z * (1 - s))
    raise ValueError(name)

In [ ]:
# Kiểm tra đạo hàm bằng sai phân hữu hạn: (f(z + ε) − f(z − ε)) / 2ε
z = np.linspace(-6, 6, 1201) + 1e-3  # lệch khỏi đúng z = 0, nơi bậc thang và ReLU gãy
for name in ["sigmoid", "tanh", "relu", "gelu", "silu"]:
    numeric = (act(name, z + 1e-6)[0] - act(name, z - 1e-6)[0]) / 2e-6
    err = np.abs(numeric - act(name, z)[1]).max()
    assert err < 1e-8, name
    print(f"{NAMES[name]:>7}: |f′ − sai phân| lớn nhất = {sci(err)}")

Đạo hàm viết tay khớp với sai phân hữu hạn tới phần tỉ. Ô dưới in các giá trị đặc trưng mà demo cũng hiển thị:
$f(0)$, $f'(0)$ và đạo hàm lớn nhất (tìm trên một lưới $z$ rất mịn từ −8 đến 8).

In [ ]:
zz = np.linspace(-8, 8, 160_001)
print(f"lưới z: {vn(len(zz), 0)} điểm, bước {vn(zz[1] - zz[0], 4)}\n")
print("   hàm    f(0)   f′(0)   f′ lớn nhất (tại z)")
for name in ["sigmoid", "tanh", "relu", "gelu", "silu"]:
    f0, d0 = act(name, 0.0)
    d = act(name, zz)[1]
    k = d.argmax()
    print(f"{NAMES[name]:>7}  {vn(f0, 3):>5}  {vn(d0, 3):>5}   {vn(d[k], 3)} (z = {vn(zz[k], 2)})")
assert act("sigmoid", 0.0)[1] == 0.25 and act("tanh", 0.0)[1] == 1.0, "khác với demo"
assert np.isclose(act("gelu", 0.0)[1], 0.5) and np.isclose(act("silu", 0.0)[1], 0.5)

Hình dưới vẽ mỗi hàm trên $z \in [-5, 5]$ như demo: $f(z)$ là đường liền màu xanh, $f'(z)$ là đường gạch màu cam.
Hãy để ý ba điều: đường gạch của bậc thang nằm sát 0; đường gạch của sigmoid không bao giờ vượt 0,25; còn đường gạch
của GELU và SiLU vượt lên trên 1 một chút.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.6), sharex=True)
zp = np.linspace(-5, 5, 1001)
for ax, name in zip(axes.flat, ["step", "sigmoid", "tanh", "relu", "gelu", "silu"]):
    f, d = act(name, zp)
    if name == "step":  # vẽ hai nửa riêng để không có đường thẳng đứng giả tại z = 0
        ax.plot(zp[zp < 0], f[zp < 0], color=BLUE, lw=2)
        ax.plot(zp[zp >= 0], f[zp >= 0], color=BLUE, lw=2, label="f(z)")
    else:
        ax.plot(zp, f, color=BLUE, lw=2, label="f(z)")
    ax.plot(zp, d, color=ORANGE, lw=2, ls="--", label="f′(z)")
    ax.axhline(0, color=GREY, lw=0.8)
    ax.axvline(0, color=GREY, lw=0.8)
    ax.set_title(NAMES[name])
    ax.set_ylim(-1.3, 2.2 if name in ("relu", "gelu", "silu") else 1.3)
    ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("z")
axes[0, 0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## 4 · Không có $f$: ba lớp gộp lại thành một

Mạng của khóa học là MLP 784 → 128 → 64 → 10 của MLSysBook: $h_1 = f(xW_1 + b_1)$, $h_2 = f(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$ (10 logit, softmax ở Bài 04). Ô dưới định nghĩa lan truyền thuận cho một $f$ bất kỳ,
với trọng số khởi tạo He như `reel/mnist_mlp/train.py` (Bài 09 giải thích vì sao chọn cỡ này).

Nếu bỏ $f$ (tức $f(z) = z$), ba phép nhân ma trận gộp lại được:

$$z = \big((xW_1 + b_1)W_2 + b_2\big)W_3 + b_3 = x\,\underbrace{W_1W_2W_3}_{784 \times 10} + \underbrace{(b_1W_2 + b_2)W_3 + b_3}_{1 \times 10}$$

Ta kiểm tra bằng số: tính cả hai vế trên 10.000 ảnh kiểm tra, với trọng số ngẫu nhiên và độ lệch khác 0.

In [ ]:
SIZES = [784, 128, 64, 10]
BATCH = 64


def he_init(sizes, rng, scale=1.0):
    """Khởi tạo He: W ~ N(0, 2/n_vào), b = 0 (như reel/mnist_mlp/train.py). `scale` dùng ở mục 11."""
    return [
        [scale * rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)]
        for n_in, n_out in itertools.pairwise(sizes)
    ]


def forward(f, params, x):
    """Lan truyền thuận. hs: đầu vào của từng lớp, phần tử cuối là 10 logit; ds: f′(z) của từng lớp ẩn."""
    hs, ds = [x], []
    for k, (W, b) in enumerate(params):
        z = hs[-1] @ W + b
        if k == len(params) - 1:
            hs.append(z)  # lớp cuối không có f
        else:
            h, d = act(f, z)
            hs.append(h)
            ds.append(d)
    return hs, ds


rng = np.random.default_rng(0)
params = he_init(SIZES, rng)
for layer in params:
    layer[1] = rng.normal(0, 0.1, layer[1].shape)  # độ lệch khác 0, để phép gộp không tầm thường
(W1, b1), (W2, b2), (W3, b3) = params
W_one = W1 @ W2 @ W3
b_one = (b1 @ W2 + b2) @ W3 + b3
z_three = forward("linear", params, Xte)[0][-1]
z_one = Xte @ W_one + b_one
gap = np.abs(z_three - z_one).max()
print(f"ma trận gộp: {W_one.shape}, độ lệch: {b_one.shape};  |z ba lớp − z một lớp| lớn nhất = {sci(gap)}")
macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
print(f"ba lớp: {vn(macs, 0)} MAC mỗi ảnh  ·  một ma trận 784 × 10: {vn(784 * 10, 0)} MAC, cùng một kết quả")
assert gap < 1e-9 and macs == 109184

Hai cách tính cho cùng 10 logit, chỉ khác ở sai số làm tròn ($10^{-15}$). Không có $f$, mạng ba lớp tốn
109.184 phép nhân–cộng nhưng chỉ là **một** phép biến đổi affine 784 → 10, đúng loại mô hình của Bài 01
(một lớp $z = xW + b$). Thêm lớp mà không thêm $f$ thì không thêm được gì ngoài chi phí.

Với $f$ = ReLU thì sao? Ô dưới tìm ma trận $A$ và độ lệch $c$ *tốt nhất* (bình phương tối thiểu) để $xA + c$ bắt chước
logit của cùng mạng ngẫu nhiên đó, lần này có ReLU, rồi đo phần không bắt chước được.

In [ ]:
def best_affine_gap(Z, X):
    """Phần phương sai mà không phép affine xA + c nào giải thích được (0 = gộp được hoàn toàn).
    Z ghép nhiều bộ 10 logit theo cột; trả về một con số cho mỗi bộ."""
    X1 = np.c_[X, np.ones(len(X))]
    R = Z - X1 @ np.linalg.lstsq(X1, Z, rcond=None)[0]
    return [
        (R[:, k : k + 10] ** 2).sum() / ((Z[:, k : k + 10] - Z[:, k : k + 10].mean(0)) ** 2).sum()
        for k in range(0, Z.shape[1], 10)
    ]


z_relu = forward("relu", params, Xte)[0][-1]
gaps = best_affine_gap(np.c_[z_three, z_relu], Xte)  # một lần giải cho cả hai bộ 10 logit
print(f"không có f: {pct(gaps[0], 4)} phương sai còn lại;  ReLU: {pct(gaps[1], 1)}")

Không có $f$, một phép affine giải thích hết (0%). Có ReLU, phép affine tốt nhất vẫn để lại 17,6% phương sai của
logit không giải thích được: mạng đã tính một hàm *không* tuyến tính của ảnh. Chính điều đó làm cho độ sâu có ý nghĩa.

## 5 · Bậc thang → sigmoid: làm mượt công tắc

Perceptron của Bài 01 dùng hàm bậc thang: một công tắc bật/tắt. Sigmoid là phiên bản “mềm” của công tắc ấy. Thật vậy,
$\sigma(z/T)$ với $T$ nhỏ dần tiến về bậc thang, và độ dốc tại 0 là $\sigma'(0)/T = 1/(4T)$: dốc hơn nhưng hẹp hơn.
Hình dưới vẽ $\sigma(z/T)$ cho $T$ = 1; 0,3; 0,1 (ba kiểu nét khác nhau) và bậc thang (xám).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
zp = np.linspace(-4, 4, 801)
ax.plot(zp[zp < 0], np.zeros((zp < 0).sum()), color=GREY, lw=3, alpha=0.6)
ax.plot(zp[zp >= 0], np.ones((zp >= 0).sum()), color=GREY, lw=3, alpha=0.6, label="bậc thang")
for T, ls, color in [(1, "-", BLUE), (0.3, "--", GREEN), (0.1, ":", ORANGE)]:
    ax.plot(zp, sigmoid(zp / T), color=color, lw=2, ls=ls, label=f"σ(z/T), T = {vn(T, 1)}")
    print(f"T = {vn(T, 1)}: độ dốc tại 0 = {vn(act('sigmoid', 0.0)[1] / T, 2)}")
ax.set(xlabel="z", ylabel="đầu ra", title="Sigmoid là bậc thang được làm mượt")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
plt.show()

### 🤔 Dự đoán trước
Lấy nơ-ron đầu tiên của lớp ẩn 1 trong mạng ngẫu nhiên ở mục 4 và tăng độ lệch $b$ của nó thêm 0,001.
Trên 10.000 ảnh kiểm tra, đầu ra của bao nhiêu ảnh thay đổi nếu nơ-ron dùng hàm bậc thang? Còn nếu dùng sigmoid?

In [ ]:
z0 = Xte @ W1[:, 0] + b1[0]  # z của nơ-ron 0 trên 10.000 ảnh
nudge = 0.001
print(f"z của nơ-ron này nằm trong [{vn(z0.min(), 2)}; {vn(z0.max(), 2)}]; tăng b thêm {vn(nudge, 3)}")
for name in ["step", "sigmoid"]:
    before, after = act(name, z0)[0], act(name, z0 + nudge)[0]
    slope = (after - before) / nudge  # độ dốc ước lượng: thay đổi đầu ra / thay đổi đầu vào
    moved = slope[slope != 0]
    print(
        f"{NAMES[name]:>9}: {vn((slope == 0).sum(), 0)} ảnh không đổi (độ dốc đúng 0); {vn(len(moved), 0)} ảnh đổi, "
        f"độ dốc từ {vn(moved.min(), 0 if name == 'step' else 2)} đến {vn(moved.max(), 0 if name == 'step' else 2)}"
    )

Với bậc thang, chỉnh $b$ một chút **không đổi gì cả** ở 9.991 ảnh: độ dốc đúng bằng 0. Ở 9 ảnh có $z$ nằm sát ngưỡng,
đầu ra nhảy hẳn từ 0 lên 1, nên “độ dốc” là $1/0{,}001$ = 1.000, và sẽ thành vô cùng nếu bước chỉnh nhỏ hơn nữa.
Cả hai đều không chỉ được hướng nào. Gradient descent (Bài 05) học bằng cách đo “chỉnh một chút thì mất mát đổi bao
nhiêu”; với bậc thang, câu trả lời là 0 ở gần như mọi nơi. Sigmoid thì đổi đầu ra của *mọi* ảnh một lượng nhỏ,
tỉ lệ với $\sigma'(z)$: luôn có một hướng để đi.

## 6 · Bão hòa: hai đầu đường cong gần như phẳng

Sigmoid và tanh mượt, nhưng đạo hàm của chúng nhỏ dần về 0 khi $|z|$ lớn: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ và
$\tanh'(z) = 1 - \tanh^2(z)$. Ta gọi một nơ-ron là **bão hòa** khi $f'(z)$ nhỏ hơn một phần mười giá trị lớn nhất của nó.
Vì $\sigma'(z) = 1/(4\cosh^2(z/2))$ và $\tanh'(z) = 1/\cosh^2 z$, ngưỡng đó có lời giải chính xác.

In [ ]:
Z_SAT = {  # |z| vượt ngưỡng này thì f′(z) < f′(0)/10
    "sigmoid": 2 * np.arccosh(np.sqrt(10)),  # cosh²(z/2) = 10
    "tanh": np.arccosh(np.sqrt(10)),  # cosh²(z) = 10
}
for name, zs in Z_SAT.items():
    assert np.isclose(act(name, zs)[1], act(name, 0.0)[1] / 10)
    print(f"{NAMES[name]:>7} bão hòa khi |z| > {vn(zs, 2)}")
print()
print("    z    σ′(z)        tanh′(z)")
for zv in [0, 1, 2, 4, 6, 8]:
    print(f"{zv:>5}    {sci(act('sigmoid', zv)[1], 2):<12} {sci(act('tanh', zv)[1], 2)}")
z = np.linspace(-6, 6, 1201)
assert np.allclose(np.tanh(z), 2 * sigmoid(2 * z) - 1)  # tanh chỉ là sigmoid được co giãn và dời xuống
print("\ntanh(z) = 2σ(2z) − 1: đúng trên cả lưới z")

Hai hàm chỉ khác nhau ở cách co giãn: $\tanh(z) = 2\sigma(2z) - 1$. Nhưng sự co giãn đó đổi hai điều quan trọng.
Đạo hàm tại 0 của tanh là 1, của sigmoid chỉ là 0,25. Và đầu ra của tanh xoay quanh 0, còn sigmoid luôn dương.
Tanh bão hòa sớm hơn (từ $|z| > 1{,}82$, so với $3{,}64$ của sigmoid), nhưng phần giữa của nó dốc gấp 4 lần.

Hình dưới tô vùng bão hòa của mỗi hàm bằng nền xám có gạch chéo. Trong vùng đó, đường gạch $f'(z)$ gần như nằm trên trục 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
zp = np.linspace(-6, 6, 1201)
for ax, name in zip(axes, ["sigmoid", "tanh"]):
    f, d = act(name, zp)
    zs = Z_SAT[name]
    for lo, hi in [(-6, -zs), (zs, 6)]:
        ax.axvspan(lo, hi, color=GREY, alpha=0.15, hatch="//", lw=0)
    ax.plot(zp, f, color=BLUE, lw=2, label="f(z)")
    ax.plot(zp, d, color=ORANGE, lw=2, ls="--", label="f′(z)")
    ax.text(4.8, -0.6, "bão hòa", ha="center", fontsize=10)
    ax.text(-4.8, -0.6, "bão hòa", ha="center", fontsize=10)
    ax.axhline(0, color=GREY, lw=0.8)
    ax.set(xlabel="z", title=f"{NAMES[name]}: bão hòa khi |z| > {vn(zs, 2)}", xlim=(-6, 6))
axes[0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

Vì sao đạo hàm nhỏ lại đáng lo? Khi lan truyền ngược (Bài 06), tín hiệu sửa lỗi bị nhân với $f'(z)$ một lần ở **mỗi**
lớp. Với sigmoid, mỗi lần nhân là nhiều nhất 0,25, nên qua $L$ lớp, riêng các thừa số đạo hàm đã nhỏ hơn $0{,}25^L$:

In [ ]:
for L in [1, 2, 5, 10, 20]:
    print(f"{L:>2} lớp sigmoid: tích các σ′ ≤ 0,25^{L} = {sci(0.25**L)}")
assert f"{0.25**20:.1e}" == "9.1e-13"  # đúng con số MLSysBook nêu cho 20 lớp

Với 20 lớp, cận trên là $9{,}1 \cdot 10^{-13}$, đúng con số chương *Neural Computation* của MLSysBook nêu. Đó mới là
trường hợp *tốt nhất*, khi mọi nơ-ron nằm đúng tại $z = 0$; nơ-ron bão hòa còn làm tín hiệu nhỏ hơn nữa.
Hiện tượng này là **gradient biến mất**, chủ đề của Bài 09. (Trọng số $W$ cũng nhân vào tín hiệu, nên một tích nhỏ
của $f'$ có thể được bù lại phần nào; Bài 09 đo điều đó trên mạng thật.)

## 7 · ReLU, GELU và SiLU: đơn giản mà hiệu quả

ReLU $= \max(0, z)$ giữ nguyên phần dương và cắt phần âm về 0. Đạo hàm của nó chỉ có hai giá trị: 1 khi $z > 0$ và
0 khi $z < 0$ (tại đúng $z = 0$ ta quy ước bằng 0). Phía dương không bao giờ bão hòa, và tính ReLU chỉ cần một phép so sánh.

GELU và SiLU là hai phiên bản mượt của ReLU, cùng dạng “$z$ nhân một cái cổng mềm trong $(0, 1)$”: GELU nhân $z$ với
$\Phi(z)$, SiLU nhân với $\sigma(z)$. Hendrycks & Gimpel (2016) đặt tên cho cả hai; Elfwing và cộng sự (2017) dùng SiLU
trong học tăng cường; Ramachandran và cộng sự (2017) tìm lại nó bằng tìm kiếm tự động và gọi là Swish,
$z\,\sigma(\beta z)$, với $\beta = 1$ chính là SiLU. Ô dưới đo ba điều: điểm thấp nhất của mỗi hàm, độ chính xác của hai
cách xấp xỉ GELU, và hình dạng của chúng gần 0.

In [ ]:
erf = np.vectorize(math.erf)
z = np.linspace(-6, 6, 12_001)
gelu_exact = 0.5 * z * (1 + erf(z / np.sqrt(2)))  # z·Φ(z), Φ là hàm phân phối của N(0, 1)
print(f"GELU dạng tanh (GPT-2, demo)  − dạng chính xác: lớn nhất {sci(np.abs(act('gelu', z)[0] - gelu_exact).max())}")
print(
    f"GELU dạng z·σ(1,702z)         − dạng chính xác: lớn nhất {sci(np.abs(z * sigmoid(1.702 * z) - gelu_exact).max())}"
)
for name in ["gelu", "silu"]:
    f = act(name, zz)[0]
    k = f.argmin()
    print(f"{NAMES[name]:>5}: nhỏ nhất {vn(f[k], 3)} tại z = {vn(zz[k], 2)}; f(−3) = {vn(act(name, -3.0)[0], 3)}")

Dạng tanh mà GPT-2 dùng chỉ lệch khỏi GELU chính xác cỡ $10^{-4}$; dạng $z\,\sigma(1{,}702z)$ thô hơn, cỡ $10^{-2}$. Khác với ReLU, GELU và SiLU
hơi âm ở phía trái (thấp nhất −0,170 và −0,278) rồi tiến dần về 0. Hình dưới phóng to quanh 0: ReLU nét liền,
GELU nét gạch, SiLU nét chấm.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
zp = np.linspace(-4, 3, 701)
for name, color, ls in [("relu", BLUE, "-"), ("gelu", ORANGE, "--"), ("silu", GREEN, ":")]:
    ax.plot(zp, act(name, zp)[0], color=color, lw=2.2, ls=ls, label=NAMES[name])
ax.axhline(0, color=GREY, lw=0.8)
ax.axvline(0, color=GREY, lw=0.8)
ax.set(xlabel="z", ylabel="f(z)", title="ReLU và hai phiên bản mượt của nó", ylim=(-0.5, 3))
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
plt.show()

Chi phí cũng khác nhau: sigmoid, tanh, GELU và SiLU cần hàm mũ, còn ReLU chỉ cần so sánh với 0. MLSysBook ghi rằng
trên phần cứng, hàm mũ được tính bằng lệnh đặc biệt, bảng tra hoặc đa thức, nên thường đắt hơn. Ô dưới đo thời gian
trên máy của bạn, với mảng cỡ 10.000 × 128 (đúng cỡ lớp ẩn 1 khi đưa cả tập kiểm tra qua mạng). Con số phụ thuộc máy;
hãy chỉ nhìn thứ tự.

In [ ]:
z_big = np.random.default_rng(1).normal(0, 2, (10_000, 128))
for name in ["relu", "sigmoid", "tanh", "gelu", "silu"]:
    t0 = time.perf_counter()
    for _ in range(5):
        act(name, z_big)
    print(f"{NAMES[name]:>7}: {vn((time.perf_counter() - t0) / 5 * 1000, 1)} ms cho f và f′")

## 8 · Giá trị $z$ thật của mạng MNIST 784 → 128 → 64 → 10

Đến giờ ta dùng $z$ trên một trục vẽ sẵn. Mạng thật cho ra những $z$ nào? Ô dưới huấn luyện lại đúng mạng tham chiếu
của khóa học (`reel/mnist_mlp/train.py`): ReLU, khởi tạo He, SGD theo lô 64 ảnh, tốc độ học $\eta = 0{,}1$,
5 lượt (epoch) qua 60.000 ảnh, seed 0, mất mát cross-entropy trên softmax. Cách tính gradient là nội dung của Bài 05–06;
ở đây chỉ cần để ý một dòng: tín hiệu đi ngược qua mỗi lớp ẩn bị **nhân với $f'(z)$**.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def train(f, eta=0.1, seed=0, n=60_000, epochs=5, sizes=SIZES, scale=1.0):
    """SGD theo lô, softmax + cross-entropy, giống reel/mnist_mlp/train.py. Trả về tham số và bản sao W lúc đầu."""
    rng = np.random.default_rng(seed)
    params = he_init(sizes, rng, scale)
    W_init = [W.copy() for W, _ in params]
    for _ in range(epochs):
        order = rng.permutation(n)  # n ảnh huấn luyện đầu tiên, xáo trộn lại mỗi lượt
        for i in range(0, n, BATCH):
            idx = order[i : i + BATCH]
            hs, ds = forward(f, params, Xtr[idx])
            dz = softmax(hs[-1])
            dz[np.arange(len(idx)), ytr[idx]] -= 1  # ∂L/∂z của 10 logit = p − onehot(y) (Bài 04)
            dz /= len(idx)
            for k in reversed(range(len(params))):  # lan truyền ngược (Bài 06)
                W, b = params[k]
                gW, gb = hs[k].T @ dz, dz.sum(0)
                if k:
                    dz = (dz @ W.T) * ds[k - 1]  # đi qua hàm kích hoạt: nhân với f′(z)
                W -= eta * gW
                b -= eta * gb
    return params, W_init


def evaluate(f, params, X=Xte, y=yte):
    z = forward(f, params, X)[0][-1]
    p = softmax(z)
    return float((z.argmax(1) == y).mean()), float(-np.log(np.maximum(p[np.arange(len(y)), y], 1e-300)).mean())

In [ ]:
t0 = time.perf_counter()
ref, _ = train("relu")
acc_ref, loss_ref = evaluate("relu", ref)
n_params = sum(W.size + b.size for W, b in ref)
print(f"mạng tham chiếu: {vn(n_params, 0)} tham số, kiểm tra đúng {pct(acc_ref)}, mất mát {vn(loss_ref, 4)}")
print(f"({vn(time.perf_counter() - t0, 1)} giây trên máy này)")
lin_full, _ = train("linear", sizes=[784, 10])  # cùng công thức, nhưng chỉ một phép affine 784 → 10 (như mục 4)
print(f"cùng công thức, hồi quy softmax 784 → 10 (không có lớp ẩn, không có f): {pct(evaluate('linear', lin_full)[0])}")
assert n_params == 109386, "khác với MLSysBook"
assert abs(acc_ref - 0.9727) < 0.0005 and abs(loss_ref - 0.0865) < 0.0005, (
    "khác với reel và demo: hãy kiểm tra lại dữ liệu"
)

97,27% trên 10.000 ảnh kiểm tra, đúng con số trong `mnist-mlp.json` mà các demo từ Bài 04 dùng, và 109.386 tham số
như MLSysBook đếm. Một phép affine duy nhất (cùng loại hàm với mạng không có $f$ ở mục 4), học theo cùng công thức,
chỉ đạt 92,04%: phần chênh lệch là thứ mà lớp ẩn và hàm kích hoạt mang lại. Giờ ta nhìn vào bên trong.

### Một ảnh đi qua lớp ẩn 1
Đưa ảnh kiểm tra đầu tiên (số 7) vào mạng. Lớp ẩn 1 có 128 nơ-ron. Sau ReLU, bao nhiêu nơ-ron cho ra **đúng 0**:
gần như không nơ-ron nào, khoảng một nửa, hay gần hết? Hãy đoán trước khi chạy.

In [ ]:
x7 = Xte[0]
z7 = x7 @ ref[0][0] + ref[0][1]  # 128 giá trị z = xW₁ + b₁ của ảnh này
logits7 = forward("relu", ref, x7[None])[0][-1][0]
print(f"nhãn {yte[0]}; mạng đoán {logits7.argmax()} (logit {vn(logits7.max(), 3)})")
print(f"{(z7 <= 0).sum()}/128 nơ-ron có z ≤ 0, nên ReLU cho ra đúng 0; z nằm trong [{vn(z7.min())}; {vn(z7.max())}]")
LOGITS_JSON = [-0.3517, -1.6476, 3.6199, 6.7722, -9.6857, 2.1441, -9.7025, 10.898, -0.8677, 2.6404]  # mnist-mlp.json
assert np.allclose(logits7, LOGITS_JSON, atol=1e-3) and logits7.argmax() == 7, "khác với reel và demo"
assert (z7 <= 0).sum() == 70, "khác với reel"

Hơn một nửa. Mỗi ảnh chỉ “bật” một phần các nơ-ron; phần còn lại im lặng hoàn toàn. Đó là **kích hoạt thưa**
(sparse), một đặc điểm Glorot và cộng sự (2011) coi là ưu điểm của ReLU.

Hình dưới đưa đúng 128 giá trị $z$ đó qua năm hàm. Đường xám là hàm; mỗi điểm là một nơ-ron: hình tròn xanh khi
$z > 0$, hình vuông cam khi $z \le 0$. Lưu ý: mạng được huấn luyện với ReLU, nên bốn ô còn lại là phép thử “nếu đưa
cùng các $z$ này qua hàm khác”, không phải một mạng đã học với hàm đó (mục 9 mới huấn luyện chúng).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.6), sharex=True)
zp = np.linspace(z7.min() - 0.3, z7.max() + 0.3, 400)
pos = z7 > 0
for ax, name in zip(axes, ["relu", "sigmoid", "tanh", "gelu", "silu"]):
    ax.plot(zp, act(name, zp)[0], color=GREY, lw=1.5)
    fz = act(name, z7)[0]
    ax.scatter(z7[pos], fz[pos], s=14, color=BLUE, marker="o", label="z > 0")
    ax.scatter(z7[~pos], fz[~pos], s=14, color=ORANGE, marker="s", label="z ≤ 0")
    ax.set_title(NAMES[name], fontsize=10)
    ax.set_xlabel("z")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("f(z)")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

Với ảnh này, mọi $z$ nằm gọn trong khoảng mà sigmoid còn dốc: không điểm nào chạm vùng phẳng ở hai đầu. Ô dưới làm
điều tương tự cho **cả** 10.000 ảnh kiểm tra và cả hai lớp ẩn (1.280.000 và 640.000 giá trị $z$).

In [ ]:
def hidden_z(f, params, X=Xte):
    """z = hW + b của từng lớp ẩn (trước hàm kích hoạt)."""
    hs, _ = forward(f, params, X)
    return [hs[k] @ params[k][0] + params[k][1] for k in range(len(params) - 1)]


zs_ref = hidden_z("relu", ref)
for k, z in enumerate(zs_ref, 1):
    p1, p50, p99 = np.percentile(z, [1, 50, 99])
    print(f"lớp ẩn {k} ({vn(z.size, 0)} giá trị): 98% nằm trong [{vn(p1)}; {vn(p99)}], trung vị {vn(p50)}")
    print(
        f"   ReLU cho đúng 0 ở {pct((z <= 0).mean(), 1)} giá trị; nơ-ron bằng 0 với cả 10.000 ảnh: {(z <= 0).all(0).sum()}"
    )
    for name, zsat in Z_SAT.items():
        print(f"   nếu đưa qua {NAMES[name]}: {pct((np.abs(z) > zsat).mean())} giá trị rơi vào vùng bão hòa")
print(
    f"ảnh số 7 ở trên: {(np.abs(z7) > Z_SAT['sigmoid']).sum()} giá trị bão hòa với sigmoid, {(np.abs(z7) > Z_SAT['tanh']).sum()} với tanh"
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(zs_ref[0].ravel(), bins=np.linspace(-8, 8, 161), color=BLUE, alpha=0.8)
ax.axvline(0, color="black", lw=1)
for name, ls in [("tanh", "--"), ("sigmoid", ":")]:
    for sign in (-1, 1):
        ax.axvline(
            sign * Z_SAT[name],
            color=ORANGE,
            lw=1.8,
            ls=ls,
            label=f"ngưỡng bão hòa của {NAMES[name]}" if sign > 0 else None,
        )
ax.set(
    xlabel="z của lớp ẩn 1",
    ylabel="số giá trị",
    title="1.280.000 giá trị z thật của lớp ẩn 1 (10.000 ảnh × 128 nơ-ron)",
    xlim=(-6, 9),  # chừa chỗ bên phải cho chú giải
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: vn(v, 0)))
ax.legend(frameon=False, loc="center right")
plt.show()

Ba điều đáng nhớ từ các con số thật:
- **Không nơ-ron nào “chết”.** ReLU cho 0 ở khoảng một nửa số giá trị, nhưng không nơ-ron nào bằng 0 với *mọi* ảnh.
  Một nơ-ron bằng 0 với ảnh này vẫn bật với ảnh khác, và gradient vẫn đến được nó qua những ảnh đó. Chỉ nơ-ron bằng 0
  với mọi đầu vào mới thật sự hết học (mục 11 tạo ra chúng).
- **Vùng bão hòa gần như trống ở lớp 1.** Chỉ 0,32% giá trị $z$ nằm ngoài hai đường chấm của sigmoid (7,98% ngoài hai
  đường gạch của tanh). Bão hòa không tự nhiên xảy ra; nó cần $z$ lớn, do trọng số khởi tạo quá lớn hoặc mạng rất sâu
  (mục 11, Bài 09).
- Lớp 2 có $z$ trải rộng hơn lớp 1, nên ở đó tanh sẽ bão hòa nhiều hơn hẳn: 27,70%.

## 9 · Đổi $f$, giữ mọi thứ khác: một thí nghiệm

Giờ ta huấn luyện cùng mạng với từng hàm. Mọi thứ khác giữ nguyên: khởi tạo He, lô 64, $\eta = 0{,}1$, cùng các seed.
Để vừa thời gian chạy, mỗi lần chỉ học **1 lượt trên 20.000 ảnh** đầu (313 bước), thay vì 5 lượt trên 60.000;
các con số vì vậy thấp hơn 97,27% của mạng tham chiếu. Mỗi hàm chạy 3 seed. Có thêm một mốc so sánh: hồi quy softmax
một lớp 784 → 10, tức một phép affine duy nhất.

### 🤔 Dự đoán trước
1. Với hàm bậc thang, $f'(z) = 0$, nên gradient không bao giờ qua được lớp ẩn: $W_1$, $W_2$ đứng yên. Mạng đó đạt khoảng
   10% (như đoán bừa), hay cao hơn hẳn?
2. Mạng ba lớp không có $f$ so với hồi quy softmax một lớp thì sao: hơn hẳn, kém hẳn, hay ngang nhau?

In [ ]:
N_SHORT, EPOCHS_SHORT = 20_000, 1
FUNCS = ["step", "sigmoid", "linear", "logreg", "tanh", "silu", "gelu", "relu"]
t0 = time.perf_counter()
runs, accs = {}, {}
for f in FUNCS:
    for seed in range(3):
        if f == "logreg":  # một lớp 784 → 10, không có lớp ẩn
            runs[f, seed] = train("linear", 0.1, seed, N_SHORT, EPOCHS_SHORT, sizes=[784, 10])
        else:
            runs[f, seed] = train(f, 0.1, seed, N_SHORT, EPOCHS_SHORT)
    accs[f] = [evaluate("linear" if f == "logreg" else f, runs[f, s][0])[0] for s in range(3)]
print(
    f"{len(FUNCS) * 3} lần huấn luyện, mỗi lần 1 lượt trên {vn(N_SHORT, 0)} ảnh = {-(-N_SHORT // BATCH)} bước "
    f"({vn(time.perf_counter() - t0, 1)} giây)\n"
)


def seed_range(a):
    return f"{pct(min(a))}–{pct(max(a))}"


def verdict(a, b):
    """Hai khoảng giá trị qua các seed chồng lên nhau thì chưa nói được bên nào tốt hơn."""
    return (
        "trong dao động giữa các seed" if max(min(a), min(b)) <= min(max(a), max(b)) else "vượt dao động giữa các seed"
    )


for f in FUNCS:
    print(f"{NAMES[f]:>17}: {' · '.join(pct(a) for a in accs[f])}   (khoảng {seed_range(accs[f])})")
moved = [np.abs(runs["step", s][0][k][0] - runs["step", s][1][k]).max() for s in range(3) for k in range(3)]
print(f"\nbậc thang: |W sau − W trước| lớn nhất của W₁, W₂, W₃ (seed 0) = {', '.join(vn(m, 3) for m in moved[:3])}")
print(f"đoán bừa (10 chữ số): {pct(1 / 10, 0)}")
print(f"không có f so với hồi quy softmax: {verdict(accs['linear'], accs['logreg'])}")
for f in ["tanh", "silu", "gelu", "relu"]:
    print(
        f"{NAMES[f]:>4} so với không có f: {verdict(accs[f], accs['linear'])}; "
        f"so với softmax 784 → 10: {verdict(accs[f], accs['logreg'])}"
    )
assert all(m == 0 for k, m in enumerate(moved) if k % 3 < 2), "W₁, W₂ của mạng bậc thang phải đứng yên"

Đọc bảng trên (nhớ rằng đây là lần học ngắn, cùng $\eta = 0{,}1$ cho mọi hàm):
- **Bậc thang** không đứng ở 10%. $W_1$ và $W_2$ đúng là đứng yên (thay đổi bằng 0), nhưng $W_3$ vẫn học, vì gradient
  đến được lớp cuối mà không phải đi qua $f'$ nào. Mạng thành một bộ phân loại tuyến tính trên 64 đặc trưng 0/1
  *ngẫu nhiên* do hai lớp đầu tạo ra lúc khởi tạo. Hàm bậc thang không chặn mọi việc học, nó chặn việc học
  **ở các lớp phía trước nó**.
- **Không có $f$** cho kết quả ngang hồi quy softmax một lớp, trong dao động giữa các seed. Ba lớp tuyến tính và một lớp
  là cùng một loại hàm (mục 4); chúng chỉ khác ở cách gradient descent đi tới lời giải, không ở thứ chúng biểu diễn được.
- **Tanh, GELU, ReLU** vượt cả hai mốc tuyến tính, vượt dao động giữa các seed. **SiLU** vượt hồi quy softmax, nhưng
  khoảng của nó chồng lên khoảng của mạng không có $f$. Với lần học ngắn này, lợi ích của $f$ chỉ vài điểm phần trăm;
  với công thức đầy đủ ở mục 8, nó là 97,27% so với 92,04%. Chênh lệch giữa bốn hàm này nhỏ.
- **Sigmoid** thì kém hẳn với cùng $\eta = 0{,}1$. Vì sao?

### 🤔 Dự đoán trước
Sigmoid kém vì các nơ-ron của nó bão hòa, hay vì lý do khác? Ô dưới đo vùng bão hòa trong chính các mạng sigmoid vừa
huấn luyện, rồi thử hai hàm tự viết, mỗi hàm chỉ đổi **một** tính chất của sigmoid:
- `tanh_quarter` $= \tanh(z)/4$: đối xứng quanh 0 như tanh, nhưng chỉ dốc 0,25 tại 0 như sigmoid;
- `sigmoid_x4` $= 4\sigma(z) - 2$: chính là sigmoid, được kéo giãn cho đối xứng quanh 0 và dốc 1 tại 0 như tanh.

(`act()` nhận cả một hàm Python trả về $f$ và $f'$, nên bạn có thể viết hàm kích hoạt của riêng mình theo cách này.)

In [ ]:
def tanh_quarter(z):
    t = np.tanh(z)
    return t / 4, (1 - t**2) / 4


def sigmoid_x4(z):
    s = sigmoid(z)
    return 4 * s - 2, 4 * s * (1 - s)


for s in range(3):
    zs = hidden_z("sigmoid", runs["sigmoid", s][0])
    print(
        f"mạng sigmoid seed {s}: bão hòa ở lớp 1 {pct((np.abs(zs[0]) > Z_SAT['sigmoid']).mean(), 3)}, "
        f"lớp 2 {pct((np.abs(zs[1]) > Z_SAT['sigmoid']).mean(), 3)}"
    )
print()
for f, label in [
    ("sigmoid", "sigmoid (dốc 0,25, luôn dương)"),
    (tanh_quarter, "tanh/4 (dốc 0,25, quanh 0)"),
    (sigmoid_x4, "4σ − 2 (dốc 1, quanh 0)"),
    ("tanh", "tanh (dốc 1, quanh 0)"),
]:
    a = (
        accs[f]
        if isinstance(f, str)
        else [evaluate(f, train(f, 0.1, s, N_SHORT, EPOCHS_SHORT)[0])[0] for s in range(3)]
    )
    print(f"{label:>31}: {seed_range(a)}")

Không phải bão hòa: trong các mạng sigmoid đã học, gần như không giá trị $z$ nào nằm trong vùng bão hòa. Thủ phạm là
**độ dốc**. Hai hàm có đạo hàm tại 0 bằng 0,25 (sigmoid và tanh/4) cùng kém; hai hàm dốc 1 (4σ − 2 và tanh) gần bằng nhau,
dù 4σ − 2 có đúng hình chữ S của sigmoid. Mỗi lần tín hiệu đi ngược qua một lớp sigmoid, nó bị nhân với nhiều nhất 0,25,
nên các lớp đầu nhận bước sửa nhỏ hơn hẳn với cùng $\eta$. Mục 11 cho thấy tăng $\eta$ bù lại được phần lớn.
Bão hòa là một vấn đề thật, nhưng nó cần $z$ lớn (mục 11) hoặc mạng sâu (Bài 09); ở mạng nông khởi tạo He, nó vắng mặt.

## 10 · Sigmoid sống tiếp làm “cổng”: SwiGLU

Trong mô hình ngôn ngữ, mỗi khối Transformer có một MLP hai lớp (Bài 17). GPT-2 và BERT dùng
$\operatorname{GELU}(xW_1 + b_1)W_2 + b_2$. Dauphin và cộng sự (2016) đề xuất **lớp có cổng** (GLU): hai phép chiếu song song,
một phép đi qua sigmoid rồi nhân từng phần tử với phép kia, $\sigma(xW) \odot xV$. Sigmoid ở đây không phải là
“kích hoạt” mà là một **cổng**: con số trong $(0, 1)$ quyết định cho bao nhiêu tín hiệu $xV$ đi qua.
Shazeer (2020) thử nhiều biến thể; biến thể dùng SiLU được gọi là SwiGLU:

$$\operatorname{FFN}_{\text{SwiGLU}}(x) = \big(\operatorname{SiLU}(xW) \odot xV\big)\,W_2, \qquad \operatorname{SiLU}(u) = u\,\sigma(u)$$

Sigmoid vẫn nằm trong đó, bên trong SiLU. Lớp này có **ba** ma trận thay vì hai, nên để giữ nguyên số tham số, Shazeer
giảm chiều ẩn $d_{ff}$ còn 2/3. Ô dưới kiểm tra con số của ông, rồi đếm phần MLP trong Llama 3 8B từ `config.json` của nó.

In [ ]:
d_model, d_ff = 768, 3072  # T5 base (Shazeer 2020, mục 3.1)
ffn_relu = 2 * d_model * d_ff  # W₁, W₂
ffn_glu = 3 * d_model * (2 * d_ff // 3)  # W, V, W₂ với d_ff = 2048
print(f"FFN thường: {vn(ffn_relu, 0)} tham số; SwiGLU với d_ff = {2 * d_ff // 3}: {vn(ffn_glu, 0)}")
assert ffn_relu == ffn_glu == 4718592

# Llama 3 8B (config.json trên Hugging Face): không độ lệch, RMSNorm, đầu ra không dùng chung với embedding
L, d, heads, kv_heads, hd, ff, vocab = 32, 4096, 32, 8, 128, 14336, 128256
attn = L * (d * (heads * hd + 2 * kv_heads * hd) + heads * hd * d)
mlp = L * 3 * d * ff  # mỗi lớp: W, V cỡ d × ff và W₂ cỡ ff × d
total = vocab * d + attn + mlp + (2 * L + 1) * d + vocab * d
print(f"Llama 3 8B: {vn(total, 0)} tham số; MLP SwiGLU chiếm {vn(mlp, 0)} ({pct(mlp / total, 1)})")
assert total == 8030261248

CONFIGS = {  # hàm kích hoạt và chiều ẩn của MLP, đọc từ config.json của từng mô hình
    "GPT-2 (2019)": ("gelu_new = GELU dạng tanh", 768, 3072),
    "BERT base (2018)": ("gelu", 768, 3072),
    "Llama 3 8B (2024)": ("silu, có cổng = SwiGLU", 4096, 14336),
    "Qwen3-8B (2025)": ("silu, có cổng = SwiGLU", 4096, 12288),
    "DeepSeek-V3 (2024)": ("silu, có cổng, mỗi chuyên gia", 7168, 2048),  # 256 chuyên gia nhỏ (Bài 24)
}
for name, (fn, dm, dff) in CONFIGS.items():
    print(f"{name:>19}: {fn:<29} d = {vn(dm, 0):>5}, d_ff = {vn(dff, 0):>6} ({vn(dff / dm, 2)} × d)")

Shazeer chọn $d_{ff} = 2048$ thay vì 3072 nên hai cách có đúng cùng số tham số. Ở Llama 3 8B, MLP SwiGLU chiếm hơn
hai phần ba toàn bộ mô hình: hàm kích hoạt nằm ở phần lớn nhất của mạng. GPT-2 và BERT dùng $d_{ff} = 4d$; các mô hình
dùng SwiGLU có chiều ẩn nhỏ hơn tương ứng (Llama 3: 3,5 × d), vì có thêm ma trận thứ ba.

Giờ ta dựng một lớp SwiGLU thật thay cho lớp ẩn 1 của mạng MNIST: $h_1 = \operatorname{SiLU}(xW) \odot xV$ với
$W, V$ cỡ 784 × 64. Hai ma trận đó có đúng $2 \cdot 784 \cdot 64$ = 100.352 trọng số, bằng $W_1$ cỡ 784 × 128 của mạng
gốc. Phần còn lại giữ nguyên ReLU: 64 → 64 → 10. Trước khi huấn luyện, ta kiểm tra gradient viết tay bằng sai phân hữu hạn.

In [ ]:
def glu_forward(params, x):
    (W, V), (W2, b2), (W3, b3) = params
    u, v = x @ W, x @ V
    g, dg = act("silu", u)
    h1 = g * v  # SiLU(xW) ⊙ xV: lớp có cổng, không độ lệch như Shazeer
    z2 = h1 @ W2 + b2
    h2 = np.maximum(0, z2)
    return (u, v, g, dg, h1, z2, h2), h2 @ W3 + b3


def glu_grads(params, x, y):
    (_, v, g, dg, h1, z2, h2), z = glu_forward(params, x)
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    W2, W3 = params[1][0], params[2][0]
    dh2 = (dz @ W3.T) * (z2 > 0)
    dh1 = dh2 @ W2.T
    # h1 = g ⊙ v: gradient chia hai nhánh; nhánh qua cổng nhân thêm SiLU′(u)
    return [x.T @ (dh1 * v * dg), x.T @ (dh1 * g), h1.T @ dh2, dh2.sum(0), h2.T @ dz, dz.sum(0)]


def glu_init(rng, h=64):
    W, V = (rng.normal(0, np.sqrt(2 / 784), (784, h)) for _ in range(2))
    return [[W, V], *he_init([h, 64, 10], rng)]


def glu_loss(params, x, y):
    p = softmax(glu_forward(params, x)[1])
    return -np.log(p[np.arange(len(y)), y]).mean()

In [ ]:
rng = np.random.default_rng(5)  # mạng SwiGLU nhỏ (h = 8) để kiểm tra gradient
P = glu_init(rng, h=8)
x, y = Xtr[:16], ytr[:16]
worst = 0.0
for arr, g in zip([a for layer in P for a in layer], glu_grads(P, x, y)):
    for _ in range(5):  # 5 phần tử ngẫu nhiên của mỗi mảng
        i = tuple(rng.integers(0, n) for n in arr.shape)
        old = arr[i]
        arr[i] = old + 1e-6
        up = glu_loss(P, x, y)
        arr[i] = old - 1e-6
        down = glu_loss(P, x, y)
        arr[i] = old
        worst = max(worst, abs((up - down) / 2e-6 - g[i]))
print(f"|gradient viết tay − sai phân| lớn nhất: {sci(worst)}")
assert worst < 1e-7

In [ ]:
def train_glu(eta=0.1, seed=0, n=N_SHORT):
    rng = np.random.default_rng(seed)
    params = glu_init(rng)
    order = rng.permutation(n)
    for i in range(0, n, BATCH):  # 1 lượt trên n ảnh, như mục 9
        idx = order[i : i + BATCH]
        for p, g in zip([a for layer in params for a in layer], glu_grads(params, Xtr[idx], ytr[idx])):
            p -= eta * g
    return params


acc_glu = []
for s in range(3):
    P = train_glu(0.1, s)
    acc_glu.append(float((glu_forward(P, Xte)[1].argmax(1) == yte).mean()))
n_glu = sum(a.size for layer in P for a in layer)
n_wv = P[0][0].size + P[0][1].size
assert n_wv == 784 * 128  # W, V cỡ 784 × 64: đúng bằng W₁ cỡ 784 × 128 của mạng gốc
print(f"W, V: {vn(n_wv, 0)} trọng số, bằng W₁ 784 × 128 của mạng gốc")
print(f"SwiGLU ({vn(n_glu, 0)} tham số): {' · '.join(pct(a) for a in acc_glu)}  (khoảng {seed_range(acc_glu)})")
print(
    f"ReLU   ({vn(n_params, 0)} tham số): {' · '.join(pct(a) for a in accs['relu'])}  (khoảng {seed_range(accs['relu'])})"
)
print(f"SwiGLU so với ReLU: {verdict(acc_glu, accs['relu'])}")
SHAZEER = {"ReLU": (1.997, 0.005), "GELU": (1.983, 0.005), "GEGLU": (1.942, 0.004), "SwiGLU": (1.944, 0.010)}
print(f"\nShazeer (2020), bảng 1: log-perplexity của T5 sau {vn(65536, 0)} bước (độ lệch chuẩn qua 4 lần chạy)")
for name, (lp, sd) in SHAZEER.items():
    print(f"   FFN {name:<6}: {vn(lp, 3)} ({vn(sd, 3)})")

Trên bài toán nhỏ này, SwiGLU **không** tốt hơn rõ rệt: khoảng kết quả của nó chồng lên khoảng của ReLU, tức trong dao
động giữa các seed, và một seed của nó kém hẳn. Điều đó không mâu thuẫn với Shazeer. Lợi ích ông đo được là trên mô hình
T5 với hàng trăm nghìn bước huấn luyện: log-perplexity giảm từ 1,997 (ReLU) xuống 1,944 (SwiGLU) sau 65.536 bước,
trong khi độ lệch chuẩn giữa các lần chạy chỉ là 0,005 và 0,010. Và chính ông viết rằng bài báo không đưa ra lời giải
thích nào cho việc các kiến trúc này hoạt động tốt, mà “quy công cho ơn trên”. Một lựa chọn được cả ngành dùng
(Llama 3, Qwen3, DeepSeek-V3) vẫn có thể là một kết quả thực nghiệm chứ chưa phải một định lý.

## 11 · Thử thay đổi

Các ô dưới chạy lại cùng hàm `train()` với tham số khác. Bạn có thể sửa giá trị rồi chạy lại. Mỗi dòng in độ chính xác
kiểm tra, cùng với số nơ-ron **chết** (bằng 0 với cả 10.000 ảnh) cho ReLU, hoặc phần $z$ nằm trong vùng bão hòa
cho sigmoid và tanh. Ba thí nghiệm:
- sigmoid với $\eta = 1$ thay vì 0,1: bù lại độ dốc nhỏ;
- ReLU với $\eta = 1$: bước quá lớn;
- tanh với trọng số khởi tạo lớn gấp 10 lần: $z$ lớn ngay từ đầu.

In [ ]:
with np.errstate(over="ignore", invalid="ignore"):  # η quá lớn có thể làm mạng phân kỳ (in ra "nan")
    for f, eta, scale in [("sigmoid", 1.0, 1), ("relu", 1.0, 1), ("tanh", 0.1, 10)]:
        print(f"{NAMES[f]}, η = {vn(eta, 1)}, khởi tạo × {scale}:")
        for seed in range(3):
            p, _ = train(f, eta, seed, N_SHORT, EPOCHS_SHORT, scale=scale)
            zs = hidden_z(f, p)
            if f == "relu":  # nơ-ron chết: z ≤ 0 với cả 10.000 ảnh
                dead = [int((z <= 0).all(0).sum()) for z in zs]
                inside = f"nơ-ron chết {dead[0]}/128 ({pct(dead[0] / 128, 1)}), {dead[1]}/64"
            else:  # bão hòa theo ngưỡng của chính hàm đó
                sat = [pct((np.abs(z) > Z_SAT[f]).mean(), 1) for z in zs]
                inside = f"bão hòa (|z| > {vn(Z_SAT[f])}) lớp 1: {sat[0]}, lớp 2: {sat[1]}"
            print(f"   seed {seed}: đúng {pct(evaluate(f, p)[0])}; {inside}")

- **Sigmoid với $\eta = 1$** vượt xa kết quả ở $\eta = 0{,}1$: độ dốc 0,25 chỉ cần một bước học lớn hơn. Trọng số lớn
  hơn cũng đẩy một ít $z$ vào vùng bão hòa (4,4–5,6% ở lớp 1), nhưng vẫn là thiểu số. So hai hàm
  kích hoạt với *cùng* $\eta$ là so cả hàm lẫn sự phù hợp của $\eta$ với hàm đó; một thí nghiệm công bằng chọn $\eta$
  riêng cho từng hàm trên tập kiểm định và học đủ lâu (Bài 07 chọn $\eta$ như vậy), nên số của nó khác với lần chạy
  ngắn ở mục 9.
- **ReLU với $\eta = 1$** làm chết nhiều nơ-ron của lớp 1. Một bước quá lớn đẩy độ lệch và trọng số tới chỗ $z \le 0$ với
  mọi ảnh; từ đó $f'(z) = 0$ với mọi ảnh, gradient không bao giờ đến nữa, và nơ-ron không thể sống lại. Ghi chú CS231n
  cảnh báo đúng điều này: tới 40% mạng có thể chết khi tốc độ học quá cao. Ở đây là hơn một phần ba lớp 1 ở mọi seed,
  và độ chính xác vừa tụt vừa dao động mạnh giữa các seed.
- **Tanh khởi tạo × 10** đưa $z$ ra vùng bão hòa và để chúng ở đó: đa số giá trị $z$ nằm ngoài $|z| > 1{,}82$
  sau khi học, và độ chính xác tụt xa. Đây là bão hòa thật, thứ mà mạng khởi tạo He ở mục 9 không gặp. Bài 09 nghiên cứu
  vì sao cỡ khởi tạo quan trọng đến vậy.

## 12 · Giới hạn và bước tiếp theo

Ta đã chọn $f$ và thấy nó quyết định mạng học được gì, nhưng chưa giải thích *vì sao* gradient lại nhân với $f'(z)$
ở mỗi lớp (Bài 05–06), và chưa thấy lớp ẩn làm gì với dữ liệu. Ví dụ nhỏ nhất là XOR, bài toán mà Bài 01 chứng minh
một nơ-ron không giải được. Sách *Deep Learning* (Goodfellow và cộng sự, mục 6.1) đưa ra một lời giải bằng ReLU với
hai nơ-ron ẩn: $W = \begin{bmatrix}1 & 1\\ 1 & 1\end{bmatrix}$, $c = (0, -1)$, $w = (1, -2)^\top$, $b = 0$.
Ô dưới chạy đúng các trọng số đó, có và không có ReLU.

In [ ]:
P4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
W_xor, c_xor, w_xor = np.array([[1.0, 1.0], [1.0, 1.0]]), np.array([0.0, -1.0]), np.array([1.0, -2.0])
for f in ["relu", "linear"]:
    out = act(f, P4 @ W_xor + c_xor)[0] @ w_xor
    print(f"{NAMES[f]:>10}: đầu ra cho (0,0), (0,1), (1,0), (1,1) = {[vn(o, 0) for o in out]}")
assert list(act("relu", P4 @ W_xor + c_xor)[0] @ w_xor) == [0, 1, 1, 0]

Có ReLU, mạng cho đúng XOR: 0, 1, 1, 0. Bỏ ReLU, cùng các trọng số đó chỉ còn là $2 - (x_1 + x_2)$: cho 2, 1, 1, 0,
và không hàm tuyến tính nào cho ra được 0, 1, 1, 0. Bài 03 mở lời giải này ra: lớp ẩn biến bốn điểm sang một không
gian mới, nơi một đường thẳng tách được chúng.

## Tóm tắt
- Không có $f$, ba lớp 784 → 128 → 64 → 10 gộp thành một ma trận 784 × 10 (sai lệch $10^{-15}$), và học ngang hồi quy
  softmax một lớp.
- Hàm bậc thang có $f' = 0$: chỉnh một chút không đổi gì, nên các lớp trước nó đứng yên; chỉ lớp cuối học được.
- $\sigma'(0) = 0{,}25$ và $\tanh'(0) = 1$; qua 20 lớp sigmoid, tích các $\sigma'$ không quá $9{,}1 \cdot 10^{-13}$.
- Trên mạng MNIST thật (97,27%), một ảnh tắt 70/128 nơ-ron ReLU ở lớp 1, nhưng không nơ-ron nào chết; vùng bão hòa của
  sigmoid gần như trống.
- Sigmoid học chậm ở mạng nông vì độ dốc nhỏ, không phải vì bão hòa; ReLU với $\eta$ quá lớn thì chết, tanh với khởi
  tạo quá lớn thì bão hòa.
- SwiGLU $= (\operatorname{SiLU}(xW) \odot xV)W_2$ chiếm hơn hai phần ba tham số của Llama 3 8B; trên MNIST nhỏ, lợi
  ích của nó nằm trong dao động giữa các seed.

## Câu hỏi ôn tập
1. Vì sao mạng ba lớp không có hàm kích hoạt chỉ là một phép affine? Ma trận gộp có cỡ bao nhiêu?
2. Mạng dùng hàm bậc thang vẫn đúng khoảng một nửa số ảnh dù $W_1$, $W_2$ không đổi. Lớp nào đã học, và học trên cái gì?
3. $\sigma'(0)$ và $\tanh'(0)$ bằng bao nhiêu? Vì sao tích của chúng qua nhiều lớp lại quan trọng?
4. Với $\eta = 0{,}1$, mạng sigmoid kém mạng tanh. Bằng chứng nào trong notebook cho thấy lý do không phải là bão hòa?
5. Nơ-ron ReLU “cho ra 0 với ảnh này” khác nơ-ron “chết” thế nào? Điều gì tạo ra nơ-ron chết?
6. Một lớp SwiGLU có mấy ma trận trọng số? Vì sao Shazeer giảm $d_{ff}$ từ 3072 xuống 2048?

<details><summary>Gợi ý đáp án</summary>

1. $((xW_1 + b_1)W_2 + b_2)W_3 + b_3 = x(W_1W_2W_3) + \text{hằng số}$; ma trận gộp cỡ 784 × 10.
2. Lớp cuối $W_3$: gradient đến nó mà không qua $f'$. Nó học một bộ phân loại tuyến tính trên 64 đặc trưng 0/1 ngẫu
   nhiên mà hai lớp đầu (đứng yên từ lúc khởi tạo) tạo ra.
3. 0,25 và 1. Lan truyền ngược nhân tín hiệu với $f'(z)$ ở mỗi lớp, nên các thừa số nhỏ hơn 1 làm tín hiệu nhỏ dần
   theo cấp số nhân (Bài 09).
4. Phần $z$ bão hòa trong các mạng sigmoid đã học gần như bằng 0; và tanh/4 (dốc 0,25) kém như sigmoid, còn 4σ − 2
   (hình sigmoid, dốc 1) gần bằng tanh. Tăng $\eta$ cũng bù lại được phần lớn.
5. Cho 0 với một ảnh là bình thường (khoảng một nửa số giá trị); nơ-ron đó vẫn học từ những ảnh làm nó bật. Nơ-ron chết
   có $z \le 0$ với *mọi* ảnh, nên $f'(z) = 0$ luôn luôn và nó không bao giờ học nữa. Tốc độ học quá lớn tạo ra chúng.
6. Ba ($W$, $V$, $W_2$) thay vì hai. $3 \cdot 768 \cdot 2048 = 2 \cdot 768 \cdot 3072$, nên so sánh giữ nguyên số tham số
   và phép tính.
</details>

## Tìm hiểu thêm
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): mạng MNIST, hàm kích
  hoạt và chi phí phần cứng của chúng.
- V. Nair & G. Hinton (2010), [Rectified Linear Units Improve Restricted Boltzmann Machines](https://www.cs.toronto.edu/~hinton/absps/reluICML.pdf).
- X. Glorot, A. Bordes & Y. Bengio (2011), [Deep Sparse Rectifier Neural Networks](https://proceedings.mlr.press/v15/glorot11a.html): ReLU và kích hoạt thưa.
- D. Hendrycks & K. Gimpel (2016), [Gaussian Error Linear Units (GELUs)](https://arxiv.org/abs/1606.08415): GELU, hai dạng xấp xỉ, và tên SiLU.
- S. Elfwing, E. Uchibe & K. Doya (2017), [Sigmoid-Weighted Linear Units](https://arxiv.org/abs/1702.03118); P. Ramachandran, B. Zoph & Q. Le (2017), [Searching for Activation Functions](https://arxiv.org/abs/1710.05941) (Swish).
- Y. Dauphin và cộng sự (2016), [Language Modeling with Gated Convolutional Networks](https://arxiv.org/abs/1612.08083) (GLU); N. Shazeer (2020), [GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202) (SwiGLU).
- Ghi chú CS231n, [Neural Networks Part 1](https://cs231n.github.io/neural-networks-1/): sigmoid, tanh, ReLU và nơ-ron chết.
- [Dive into Deep Learning, mục 5.1](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html) và [Deep Learning, chương 6](https://www.deeplearningbook.org/contents/mlp.html).
- A. Karpathy, [Building makemore Part 3: Activations & Gradients, BatchNorm](https://youtu.be/P6sfmUTpUmc): tanh bão hòa trong một mạng thật.

**Bài tiếp theo:** [Bài 03 · Mạng nhiều lớp & bài toán XOR](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/): lớp ẩn biến dữ liệu sang không gian mới, nơi XOR tách được.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")